In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
workspace_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'shared').exists() and (path / 'artifacts').exists()
)
artifacts_root = workspace_root / 'artifacts' / 'ps-004-headcount-forecasting'
data_path = workspace_root / 'shared' / 'data' / '4_processed' / 'workforce_clean.parquet'
df = pl.read_parquet(data_path)
all_sector_df = df.filter(pl.col('sector') == 'All').sort(['profession', 'year'])
print(f"Filtered sector=='All' shape: {all_sector_df.shape}")
print(all_sector_df.head())
if all_sector_df.is_empty():
    print("No sector='All' rows found in the canonical parquet. Falling back to year-profession totals summed across sectors so the EDA can run.")
    analysis_df = (
        df.group_by(['year', 'profession'])
        .agg(pl.col('count').sum().alias('count'))
        .sort(['profession', 'year'])
    )
else:
    analysis_df = all_sector_df.select(['year', 'profession', 'count']).sort(['profession', 'year'])
print(f"Analysis shape: {analysis_df.shape}")
print(analysis_df.head())

In [ ]:
figure_path = artifacts_root / 'reports' / 'figures' / 'headcount_time_series.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
plot_df = analysis_df.to_pandas()
plt.figure(figsize=(10, 6))
for profession in sorted(plot_df['profession'].unique()):
    subset = plot_df[plot_df['profession'] == profession].sort_values('year')
    plt.plot(subset['year'], subset['count'], marker='o', linewidth=2, label=profession)
plt.title('Headcount Over Time by Profession')
plt.xlabel('Year')
plt.ylabel('Headcount')
plt.grid(alpha=0.3)
plt.legend(title='Profession')
plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure written to {figure_path}")

In [ ]:
year_range_summary = (
    analysis_df.group_by('profession')
    .agg(
        pl.col('year').min().alias('min_year'),
        pl.col('year').max().alias('max_year'),
        pl.len().alias('row_count')
    )
    .sort('profession')
)
print(year_range_summary)

In [ ]:
diff_summary = (
    analysis_df.sort(['profession', 'year'])
    .with_columns(pl.col('count').diff().over('profession').alias('first_diff'))
    .drop_nulls('first_diff')
    .group_by('profession')
    .agg(
        pl.col('first_diff').mean().round(2).alias('mean_diff'),
        pl.col('first_diff').std().round(2).alias('std_diff')
    )
    .sort('profession')
)
print(diff_summary)

In [ ]:
autocorrelation_rows = []
for profession in analysis_df.select('profession').unique().sort('profession').to_series().to_list():
    series = (
        analysis_df.filter(pl.col('profession') == profession)
        .sort('year')
        .get_column('count')
        .cast(pl.Float64)
    )
    current = series.slice(1)
    lagged = series.slice(0, len(series) - 1)
    corr = pl.DataFrame({'current': current, 'lagged': lagged}).select(pl.corr('current', 'lagged')).item()
    autocorrelation_rows.append({
        'profession': profession,
        'lag_1_autocorrelation': round(corr, 4) if corr is not None else None,
    })
autocorrelation_summary = pl.DataFrame(autocorrelation_rows).sort('profession')
print(autocorrelation_summary)

In [ ]:
summary_table = (
    analysis_df.sort(['profession', 'year'])
    .group_by('profession')
    .agg(
        pl.col('count').min().alias('min_headcount'),
        pl.col('count').max().alias('max_headcount'),
        pl.col('count').mean().round(2).alias('mean_headcount'),
        pl.col('count').first().alias('first_year_headcount'),
        pl.col('count').last().alias('last_year_headcount')
    )
    .with_columns(
        pl.when(pl.col('last_year_headcount') > pl.col('first_year_headcount'))
        .then(pl.lit('upward'))
        .otherwise(pl.lit('not upward'))
        .alias('trend_direction')
    )
    .select(['profession', 'min_headcount', 'max_headcount', 'mean_headcount', 'trend_direction'])
    .sort('profession')
)
print(summary_table)